<h1>Getting started with the ufs-community/ufs-analysis package</h1>
<h3>Aug. 2026</h3>

This notebook demonstrates how to use the ```ufs-community/ufs-analysis``` package located here: https://github.com/ufs-community/ufs-analysis.<br>

<h2>Prerequisite</h2>

The `ufs-analysis` package is engineered to work on Linux computing platforms.  It also requires a highly specific Conda environment to be active in your computing environment.  The required packages, and their versions, are defined in the `environment.yml` file located in the top-most directory in the repository.  You must create and activate this exact Conda environment before using `ufs-analysis`.

<h2>ufs-analysis</h2>

The `ufs-analysis` package has 3 sub-packages: `datareader`, `regridder`, and `util`.  Here we focus on the first two, as `util` is responsible for highly specialized analysis codes beyond the scope of this tutorial.

<h2>DataReader class</h2>

The DataReader class is used to package model and verification data into a unifying, canonical data structure.  This class is the entry-point for all downstream functionality in the `ufs-analysis` package.  It offers three core pieces of functionality:<br>1. Getting model and verification datasets from remote file systems like Amazon S3 or Google Cloud Storage buckets,<br>2. Standardizing the coordinate system, and<br>3. Retrieving subsets of the data.<br><br>
One requirement is that datasets are assumed to exist as `zarr` files in their remote locations. Under the hood, the `ufs-analysis` package stores data as Xarray Datasets and DataArrays.<br><br>
Let's step through an example in which we query 1 UFS dataset and 1 ERA5 dataset.
<br><br>

<h2>DataReader example</h2>

First, you must define where the `ufs-analysis` source code is located.  The `basedir` variable should point to the root directory of the package.  For example, if this tutorial notebook is saved in the same parent directory as `ufs-analysis`, then set `basedir` as:

In [1]:
basedir = './ufs-analysis'

Now we can import our code:

In [2]:
import os
import sys

# Point to root directory of repository
root_dir = os.path.join(os.getcwd(), basedir)
if root_dir not in sys.path:
    sys.path.insert(0, root_dir)

# Import datareader module
from src.datareader import datareader as dr

As of Aug. 2026, the DataReader class has two built-in data source options: `UFS` and `ERA5`.  Programmatically speaking, these data sources are handled as sub-classes to the `DataReader` super class, with their only uniqueness being each particular implementation of Xarray's `.open_zarr()` method.  Users need not be aware of these details; the point here is that developers of the `ufs-analysis` package can easily define additional subclasses to include more data sources (for example MERRA2) without interfering with the `DataReader` superclass, provided that a remote data source is indeed available for the new model.

In [3]:
# Get SFS atmospheric model data
ufs_data_reader = dr.getDataReader(datasource='UFS',                  
                                   experiment = 'baseline',
                                   model='atm')

No filename provided; deferring to default
Reading data from s3://noaa-oar-sfsdev-pds/experiments/phase_1/baseline/atm_monthly.zarr


Here, the `experiment` parameter is special to UFS.  By default, a UFS-type `DataReader` pulls SFS experimental data from `s3://noaa-oar-sfsdev-pds/experiments/phase_1/<experiment>/atm_monthly.zarr`.<br>

If you navigate to the webpage for NOAA's S3 bucket here:<br>
`https://noaa-oar-sfsdev-pds.s3.amazonaws.com/index.html`<br>
you'll see that `phase_1` has `baseline`, `beta.0.1`, `c96_beta.0.1`, and `cpc_ics` experiments are available.<br><br>This data bucket is subject to change as new experiments are undertaken and new data published.  For added control, an alternative for specifying datasets is to use the `filename` parameter.  Here is an example of grabbing the same dataset using `filename`:


In [4]:
# Get SFS atmospheric model data
ufs_data_reader = dr.getDataReader(datasource='UFS',                  
                                   filename=f'experiments/phase_1/baseline/atm_monthly.zarr',
                                   model='atm')

print("\nThis data reader is type:", type(ufs_data_reader))

Reading data from s3://noaa-oar-sfsdev-pds/experiments/phase_1/baseline/atm_monthly.zarr

This data reader is type: <class 'src.datareader.UFS_DataReader.UFS_DataReader'>


For UFS, the base url is set to `s3://noaa-oar-sfsdev-pds/` onto which the `filename` parameter is appended.

The `model` parameter can be one of `atm`, `ocn`, `lnd`, `ice`, or `wav`.  As of Aug. 2026, this parameter is not truly active (i.e. does not actually do anything), but that will change as the `ufs-analysis` package is further developed for non-`atm` models.


Now let's get an ERA5 verification dataset:

In [5]:
# Get ERA5 Analysis-Ready data
era5_data_reader = dr.getDataReader(datasource='ERA5')

print("\nThis data reader is type:", type(era5_data_reader))

No filename provided; deferring to default
Reading data from gs://gcp-public-data-arco-era5/ar/1959-2022-6h-512x256_equiangular_conservative.zarr

This data reader is type: <class 'src.datareader.ERA5_DataReader.ERA5_DataReader'>


The ERA5 DataReader has a base URL fixed to `gs://gcp-public-data-arco-era5/ar/`.  Here, "ar" stands for "Analysis-Ready" as these datasets come pre-processed on standard rectilinear grids.<br><br>
ERA5's cloud-based datastore also publish "Cloud-Optimized" datasets which are the raw data used to produce the Analysis-Ready datasets.  For "co" datasets, parameters are represented by their native grid resolution.<br><br>
See https://github.com/google-research/arco-era5 for more information, and https://console.cloud.google.com/storage/browser/gcp-public-data-arco-era5?inv=1&invt=Ab1EaQfor for a complete index of ERA5's cloud bucket.<br><br>

Use the `filename` argument to specify a particular ERA5 dataset to extract.  In this example, the same default dataset is specified:

In [6]:
# Get ERA5 Analysis-Ready data
era5_data_reader = dr.getDataReader(datasource='ERA5',
                                    filename='1959-2022-6h-512x256_equiangular_conservative.zarr')

Reading data from gs://gcp-public-data-arco-era5/ar/1959-2022-6h-512x256_equiangular_conservative.zarr


<h2>Coordinate Standardization</h2>

Every dataset queried by the `DataReader` class has its coordinate system standardized as so:<br>
- Coordinate names of 'latitude' or 'y' are renamed to 'lat'
- Coordinate names of 'longitude' or 'x' are renamed to 'lon'
- Coordinate name of 'level' is renamed to 'lev'
- Longitudes are sorted in ascending order
- Latitudes are sorted in descending order
- The dimensionality of the underlying data arrays are ordered as ***[time, init, lead, lat, lon, member]*** depending on which coordinates are present

<h2>DataReader Methods</h2>

The `DataReader` class has convenvient built-in methods for summarizing and exploring your data.

The `.dataset_url()` method prints the url of your dataset:

In [7]:
ufs_data_reader.dataset_url()

's3://noaa-oar-sfsdev-pds/experiments/phase_1/baseline/atm_monthly.zarr'

The `.info()` method prints Xarray's default dataset summary:

In [8]:
ufs_data_reader.info()

<xarray.Dataset> Size: 216GB
Dimensions:                (init: 60, member: 11, lead: 4, lat: 192, lon: 384,
                            lev: 39, depthBelowLandLayer: 4)
Coordinates:
  * init                   (init) datetime64[ns] 480B 1994-05-01 ... 2023-11-01
  * member                 (member) int64 88B 0 1 2 3 4 5 6 7 8 9 10
  * lead                   (lead) int64 32B 0 1 2 3
  * lat                    (lat) float64 2kB 89.28 88.36 87.42 ... -88.36 -89.28
  * lon                    (lon) float64 3kB 0.0 0.9375 1.875 ... 358.1 359.1
  * lev                    (lev) float64 312B 0.01 0.02 0.04 ... 975.0 1e+03
  * depthBelowLandLayer    (depthBelowLandLayer) float64 32B 0.0 0.1 0.4 1.0
    atmosphereSingleLayer  float64 8B ...
    heightAboveGround      float64 8B ...
    meanSea                float64 8B ...
    nominalTop             float64 8B ...
    surface                float64 8B ...
Data variables: (12/40)
    albdo                  (init, lead, member, lat, lon) float32 779M

The `.list_variables()` method prints a concise list of every variable present in the dataset:

In [9]:
ufs_data_reader.list_variables()

Available variables:
                              
albdo   phi      soilt  taux  
capesfc prate    soilw  tauy  
gflux   pwat     spfh   tcc   
lwdsfc  siconc   spfh2m tmp2m 
lwusfc  sithick  ssrun  tmpsfc
lwutoa  slp      swdsfc tozne 
mslhf   snod     swdtoa tprs  
msshf   snowc    swe    uprs  
o3mr    snowfall swusfc vprs  
pevpr   soill    swutoa watr  


The `.describe()` method prints a Pandas table of each variable and its dimensions, shape, description, and units.

In [10]:
ufs_data_reader.describe()

,Variable,Dimensions,Shape,Description,Units
0,albdo,"init, lead, member, lat, lon",60 × 4 × 11 × 192 × 384,Forecast albedo,%
1,capesfc,"init, lead, member, lat, lon",60 × 4 × 11 × 192 × 384,Convective available potential energy,J kg**-1
2,gflux,"init, lead, member, lat, lon",60 × 4 × 11 × 192 × 384,Ground heat flux,W m**-2
3,lwdsfc,"init, lead, member, lat, lon",60 × 4 × 11 × 192 × 384,Surface downward long-wave radiation flux,W m**-2
4,lwusfc,"init, lead, member, lat, lon",60 × 4 × 11 × 192 × 384,Surface upward long-wave radiation flux,W m**-2
5,lwutoa,"init, lead, member, lat, lon",60 × 4 × 11 × 192 × 384,Surface upward long-wave radiation flux,W m**-2
6,mslhf,"init, lead, member, lat, lon",60 × 4 × 11 × 192 × 384,Mean surface latent heat flux,W m**-2
7,msshf,"init, lead, member, lat, lon",60 × 4 × 11 × 192 × 384,Mean surface sensible heat flux,W m**-2
8,o3mr,"init, lead, member, lat, lon, lev",60 × 4 × 11 × 192 × 384 × 39,Ozone mixing ratio,kg kg**-1
9,pevpr,"init, lead, member, lat, lon",60 × 4 × 11 × 192 × 384,unknown,unknown


You can also run the `.describe()` method on a particular variable:

In [11]:
ufs_data_reader.describe('tprs')


Variable: tprs
Dimensions: ('init', 'lead', 'member', 'lat', 'lon', 'lev')
Shape: (60, 4, 11, 192, 384, 39)
Attributes:
  - long_name: Temperature
  - units: K


The `.dataset()` method returns the underlying Xarray dataset in case you want to bypass our canonical data structure in favor of Xarray's functionality:

In [12]:
ds = ufs_data_reader.dataset()
ds.lead.values

array([0, 1, 2, 3])

<h2>DataReader.retrieve()</h2>

The `DataReader.retrieve()` method takes subsets of your data while preserving the complete underlying dataset.<br>
It can also compute mean or standard deviation across any dimension or combination of dimensions.  The retrieved dataset can be written to a `netcdf` file on your local machine by specifying a full path name ending in either `.nc` or `.csv`. (But be careful writing Xarray datasets to disk -- Ensure your data do not exceed practical size limits on your file system.)

Here is the call signature:<br>

**var:** *Union[str, List[str]]*<br>
**lat:** *Union[float, Tuple[float, float]] = None*<br>
**lon:** *Union[float, Tuple[float, float]] = None*<br>
**time:** *Union[datetime.datetime, str, Tuple] = None*<br>
**initmonths:** *Union[int, Tuple, list] = None*<br>
**lev:** *Union[float, Tuple[float, float]] = None*<br>
**depth:** *Union[float, Tuple[float, float]] = None*<br>
**member:** *Union[int, Tuple[int, int]] = None*<br>
**lead:** *Union[int, Tuple[int, int]] = None*<br>
**ens_avg:** *bool = False*<br>

**mean:** *Union[str, List[str]] = None*<br>
**std:** *Union[str, List[str]] = None*<br>
**save_path:** *str = None*<br>

The only required parameter for a `DataReader.retrieve()` is `var`, which can be a single string value like `['temperature']` or a list of string values like `['temperature', 'pressure']`.  The remaining parameters are all disabled (i.e. `None`) by default.

<h2>Retrieve Examples</h2>

In [13]:
ds = ufs_data_reader.retrieve(var='tprs',     # or select a LIST of fields like ['tprs', 'tmp2m', 'uprs', 'vprs']
                              lat=(30, -30),  # or (-30, 30) to same effect
                              lon=(210, 250), # or (250, 210) to same effect
                              initmonths=11,  # or (5, 11) for multiple specific inits
                              lev=500,        # or (500, 700) for a slice of levs
                              member=0,       # or (0, 5) for a slice of members
                              lead=1)         # or (0, 2) for a slice of leads

ds

<xarray.Dataset> Size: 331kB
Dimensions:                (init: 30, lead: 1, member: 1, lat: 64, lon: 43,
                            lev: 1)
Coordinates:
  * init                   (init) datetime64[ns] 240B 1994-11-01 ... 2023-11-01
  * lead                   (lead) int64 8B 1
  * member                 (member) int64 8B 0
  * lat                    (lat) float64 512B 29.45 28.52 ... -28.52 -29.45
  * lon                    (lon) float64 344B 210.0 210.9 211.9 ... 248.4 249.4
  * lev                    (lev) float64 8B 500.0
    atmosphereSingleLayer  float64 8B 0.0
    heightAboveGround      float64 8B 2.0
    meanSea                float64 8B 0.0
    nominalTop             float64 8B 0.0
    surface                float64 8B 0.0
Data variables:
    tprs                   (init, lead, member, lat, lon, lev) float32 330kB dask.array<chunksize=(1, 1, 1, 64, 43, 1), meta=np.ndarray>
Attributes:
    Conventions:             CF-1.7
    GRIB_centre:             kwbc
    GRIB_centreDescription:  US National Weather Service - NCEP
    history:                 2024-12-09T11:56 GRIB to CDM+CF via cfgrib-0.9.1...
    institution:             US National Weather Service - NCEP

Note that all bounds are *inclusive*.

Temporal dimensions are handled differently depending on whether a model dataset (with `init` and `lead` dimensions) or a verification dataset (with `time` dimension) is provided.  Note that datetimes must be specified as 'YYYY-MM-DD' strings.

For a model dataset with `init` and `lead`, the `time` parameter subsets data by `init`:

In [14]:
ds = ufs_data_reader.retrieve(var='tprs',
                              time=('1994-05-01', '1997-11-01'))

ds

<xarray.Dataset> Size: 4GB
Dimensions:                (init: 8, lead: 4, member: 11, lat: 192, lon: 384,
                            lev: 39)
Coordinates:
  * init                   (init) datetime64[ns] 64B 1994-05-01 ... 1997-11-01
  * lead                   (lead) int64 32B 0 1 2 3
  * member                 (member) int64 88B 0 1 2 3 4 5 6 7 8 9 10
  * lat                    (lat) float64 2kB 89.28 88.36 87.42 ... -88.36 -89.28
  * lon                    (lon) float64 3kB 0.0 0.9375 1.875 ... 358.1 359.1
  * lev                    (lev) float64 312B 0.01 0.02 0.04 ... 975.0 1e+03
    atmosphereSingleLayer  float64 8B ...
    heightAboveGround      float64 8B ...
    meanSea                float64 8B ...
    nominalTop             float64 8B ...
    surface                float64 8B ...
Data variables:
    tprs                   (init, lead, member, lat, lon, lev) float32 4GB dask.array<chunksize=(1, 1, 11, 192, 384, 1), meta=np.ndarray>
Attributes:
    Conventions:             CF-1.7
    GRIB_centre:             kwbc
    GRIB_centreDescription:  US National Weather Service - NCEP
    history:                 2024-12-09T11:56 GRIB to CDM+CF via cfgrib-0.9.1...
    institution:             US National Weather Service - NCEP

In [15]:
ds.init.values

array(['1994-05-01T00:00:00.000000000', '1994-11-01T00:00:00.000000000',
       '1995-05-01T00:00:00.000000000', '1995-11-01T00:00:00.000000000',
       '1996-05-01T00:00:00.000000000', '1996-11-01T00:00:00.000000000',
       '1997-05-01T00:00:00.000000000', '1997-11-01T00:00:00.000000000'],
      dtype='datetime64[ns]')

Whereas for a verification dataset, the `time` parameter subsets data by its `time` dimension:

In [16]:
ds = era5_data_reader.retrieve(var='temperature',
                               time=('1994-05-01', '1997-11-01'))

ds

<xarray.Dataset> Size: 35GB
Dimensions:      (time: 5121, lat: 256, lon: 512, lev: 13)
Coordinates:
  * time         (time) datetime64[ns] 41kB 1994-05-01 ... 1997-11-01
  * lat          (lat) float64 2kB 89.65 88.95 88.24 ... -88.24 -88.95 -89.65
  * lon          (lon) float64 4kB 0.0 0.7031 1.406 2.109 ... 357.9 358.6 359.3
  * lev          (lev) int64 104B 50 100 150 200 250 ... 600 700 850 925 1000
Data variables:
    temperature  (time, lat, lon, lev) float32 35GB dask.array<chunksize=(124, 256, 512, 13), meta=np.ndarray>

In [17]:
ds.time.values

array(['1994-05-01T00:00:00.000000000', '1994-05-01T06:00:00.000000000',
       '1994-05-01T12:00:00.000000000', ...,
       '1997-10-31T12:00:00.000000000', '1997-10-31T18:00:00.000000000',
       '1997-11-01T00:00:00.000000000'], dtype='datetime64[ns]')

Back to the UFS dataset...

Calculating ensemble average:

The `ens_avg=True` parameter is specific to the `member` dimension.  By default this is set to `False`.

In [18]:
ds1 = ufs_data_reader.retrieve(var='tprs',
                              lat=(30, -30),
                              lon=(210, 250),
                              initmonths=(5, 11),
                              members=(0, 5),
                              lev=(500, 700),
                              ens_avg=True)  # or use mean=['member'] to same effect. (see next cell)
ds1

Taking Ensemble Average


<xarray.Dataset> Size: 13MB
Dimensions:                (init: 60, lead: 4, lat: 64, lon: 43, lev: 5)
Coordinates:
  * init                   (init) datetime64[ns] 480B 1994-05-01 ... 2023-11-01
  * lead                   (lead) int64 32B 0 1 2 3
  * lat                    (lat) float64 512B 29.45 28.52 ... -28.52 -29.45
  * lon                    (lon) float64 344B 210.0 210.9 211.9 ... 248.4 249.4
  * lev                    (lev) float64 40B 500.0 550.0 600.0 650.0 700.0
    atmosphereSingleLayer  float64 8B 0.0
    heightAboveGround      float64 8B 2.0
    meanSea                float64 8B 0.0
    nominalTop             float64 8B 0.0
    surface                float64 8B 0.0
Data variables:
    tprs                   (init, lead, lat, lon, lev) float32 13MB dask.array<chunksize=(1, 1, 64, 43, 1), meta=np.ndarray>
Attributes:
    Conventions:             CF-1.7
    GRIB_centre:             kwbc
    GRIB_centreDescription:  US National Weather Service - NCEP
    history:                 2024-12-09T11:56 GRIB to CDM+CF via cfgrib-0.9.1...
    institution:             US National Weather Service - NCEP

Note that `ens_avg` is calculated *after* subsetting takes place.  So, if you specify members to subset, then `ens_avg` will only consider those members in its calculation.  The same is true for the next cell.

Using the `mean=[dimension1, dimension2, ...]` parameter is generic across any dimension.

In [19]:
ds2 = ufs_data_reader.retrieve(var='tprs',
                              lat=(30, -30),
                              lon=(210, 250),
                              initmonths=(5, 11),
                              members=(0, 5),
                              lev=(500, 700),
                              mean=['member'])  # Or use ens_avg=True to same effect. (see previous cell)

ds2

Calculating MEAN


<xarray.Dataset> Size: 13MB
Dimensions:                (init: 60, lead: 4, lat: 64, lon: 43, lev: 5)
Coordinates:
  * init                   (init) datetime64[ns] 480B 1994-05-01 ... 2023-11-01
  * lead                   (lead) int64 32B 0 1 2 3
  * lat                    (lat) float64 512B 29.45 28.52 ... -28.52 -29.45
  * lon                    (lon) float64 344B 210.0 210.9 211.9 ... 248.4 249.4
  * lev                    (lev) float64 40B 500.0 550.0 600.0 650.0 700.0
    atmosphereSingleLayer  float64 8B 0.0
    heightAboveGround      float64 8B 2.0
    meanSea                float64 8B 0.0
    nominalTop             float64 8B 0.0
    surface                float64 8B 0.0
Data variables:
    tprs                   (init, lead, lat, lon, lev) float32 13MB dask.array<chunksize=(1, 1, 64, 43, 1), meta=np.ndarray>
Attributes:
    Conventions:             CF-1.7
    GRIB_centre:             kwbc
    GRIB_centreDescription:  US National Weather Service - NCEP
    history:                 2024-12-09T11:56 GRIB to CDM+CF via cfgrib-0.9.1...
    institution:             US National Weather Service - NCEP

We can see that the results for both queries are identical:

In [20]:
ds1.identical(ds2)

True

We can use the same `mean` parameter to calculate mean across a lat-lon region:

In [21]:
ds = ufs_data_reader.retrieve(var='tprs',
                              lat=(30, -30),
                              lon=(210, 250),
                              initmonths=11,
                              lev=500,
                              member=(0, 5),
                              lead=1,
                              mean=['lat', 'lon'])

ds

Calculating MEAN
Calculating MEAN


<xarray.Dataset> Size: 1kB
Dimensions:                (init: 30, lead: 1, member: 6, lev: 1)
Coordinates:
  * init                   (init) datetime64[ns] 240B 1994-11-01 ... 2023-11-01
  * lead                   (lead) int64 8B 1
  * member                 (member) int64 48B 0 1 2 3 4 5
  * lev                    (lev) float64 8B 500.0
    atmosphereSingleLayer  float64 8B 0.0
    heightAboveGround      float64 8B 2.0
    meanSea                float64 8B 0.0
    nominalTop             float64 8B 0.0
    surface                float64 8B 0.0
Data variables:
    tprs                   (init, lead, member, lev) float32 720B dask.array<chunksize=(1, 1, 6, 1), meta=np.ndarray>
Attributes:
    Conventions:             CF-1.7
    GRIB_centre:             kwbc
    GRIB_centreDescription:  US National Weather Service - NCEP
    history:                 2024-12-09T11:56 GRIB to CDM+CF via cfgrib-0.9.1...
    institution:             US National Weather Service - NCEP

Calculate standard deviation across lat-lon region in the same fashion:

In [22]:
ds = ufs_data_reader.retrieve(var='tprs',
                              lat=(30, -30),
                              lon=(210, 250),
                              initmonths=11,
                              lev=500,
                              member=(0, 5),
                              lead=1,
                              std=['lat', 'lon'])

ds

Calculating STD


<xarray.Dataset> Size: 1kB
Dimensions:                (init: 30, lead: 1, member: 6, lev: 1)
Coordinates:
  * init                   (init) datetime64[ns] 240B 1994-11-01 ... 2023-11-01
  * lead                   (lead) int64 8B 1
  * member                 (member) int64 48B 0 1 2 3 4 5
  * lev                    (lev) float64 8B 500.0
    atmosphereSingleLayer  float64 8B 0.0
    heightAboveGround      float64 8B 2.0
    meanSea                float64 8B 0.0
    nominalTop             float64 8B 0.0
    surface                float64 8B 0.0
Data variables:
    tprs                   (init, lead, member, lev) float32 720B dask.array<chunksize=(1, 1, 6, 1), meta=np.ndarray>
Attributes:
    Conventions:             CF-1.7
    GRIB_centre:             kwbc
    GRIB_centreDescription:  US National Weather Service - NCEP
    history:                 2024-12-09T11:56 GRIB to CDM+CF via cfgrib-0.9.1...
    institution:             US National Weather Service - NCEP

After taking subsets, the complete underlying dataset is preserved!

In [23]:
ufs_data_reader.dataset()

<xarray.Dataset> Size: 216GB
Dimensions:                (init: 60, member: 11, lead: 4, lat: 192, lon: 384,
                            lev: 39, depthBelowLandLayer: 4)
Coordinates:
  * init                   (init) datetime64[ns] 480B 1994-05-01 ... 2023-11-01
  * member                 (member) int64 88B 0 1 2 3 4 5 6 7 8 9 10
  * lead                   (lead) int64 32B 0 1 2 3
  * lat                    (lat) float64 2kB 89.28 88.36 87.42 ... -88.36 -89.28
  * lon                    (lon) float64 3kB 0.0 0.9375 1.875 ... 358.1 359.1
  * lev                    (lev) float64 312B 0.01 0.02 0.04 ... 975.0 1e+03
  * depthBelowLandLayer    (depthBelowLandLayer) float64 32B 0.0 0.1 0.4 1.0
    atmosphereSingleLayer  float64 8B ...
    heightAboveGround      float64 8B ...
    meanSea                float64 8B ...
    nominalTop             float64 8B ...
    surface                float64 8B ...
Data variables: (12/40)
    albdo                  (init, lead, member, lat, lon) float32 779MB dask.array<chunksize=(1, 1, 11, 192, 384), meta=np.ndarray>
    capesfc                (init, lead, member, lat, lon) float32 779MB dask.array<chunksize=(1, 1, 11, 192, 384), meta=np.ndarray>
    gflux                  (init, lead, member, lat, lon) float32 779MB dask.array<chunksize=(1, 1, 11, 192, 384), meta=np.ndarray>
    lwdsfc                 (init, lead, member, lat, lon) float32 779MB dask.array<chunksize=(1, 1, 11, 192, 384), meta=np.ndarray>
    lwusfc                 (init, lead, member, lat, lon) float32 779MB dask.array<chunksize=(1, 1, 11, 192, 384), meta=np.ndarray>
    lwutoa                 (init, lead, member, lat, lon) float32 779MB dask.array<chunksize=(1, 1, 11, 192, 384), meta=np.ndarray>
    ...                     ...
    tmpsfc                 (init, lead, member, lat, lon) float32 779MB dask.array<chunksize=(1, 1, 11, 192, 384), meta=np.ndarray>
    tozne                  (init, lead, member, lat, lon) float32 779MB dask.array<chunksize=(1, 1, 11, 192, 384), meta=np.ndarray>
    tprs                   (init, lead, member, lat, lon, lev) float32 30GB dask.array<chunksize=(1, 1, 11, 192, 384, 1), meta=np.ndarray>
    uprs                   (init, lead, member, lat, lon, lev) float32 30GB dask.array<chunksize=(1, 1, 11, 192, 384, 1), meta=np.ndarray>
    vprs                   (init, lead, member, lat, lon, lev) float32 30GB dask.array<chunksize=(1, 1, 11, 192, 384, 1), meta=np.ndarray>
    watr                   (init, lead, member, lat, lon) float32 779MB dask.array<chunksize=(1, 1, 11, 192, 384), meta=np.ndarray>
Attributes:
    Conventions:             CF-1.7
    GRIB_centre:             kwbc
    GRIB_centreDescription:  US National Weather Service - NCEP
    history:                 2024-12-09T11:56 GRIB to CDM+CF via cfgrib-0.9.1...
    institution:             US National Weather Service - NCEP

<h2>The SUPPLIED DataReader</h2>

There is a 3rd type of DataReader called the `SUPPLIED_DataReader`.  The `SUPPLIED_DataReader` can wrap its functionality around **any** Xarray dataset that you provide to it.  Here is a dummy example in which we take the dataset `ds` and "supply" it to a new `DataReader` object:

In [24]:
ds = ufs_data_reader.retrieve(var='tprs',
                              time=('1994-05-01', '1997-11-01'))

supplied_data_reader = dr.getDataReader(datasource='SUPPLIED',
                                        dataset=ds)

In [25]:
print(type(supplied_data_reader))

<class 'src.datareader.SUPPLIED_DataReader.SUPPLIED_DataReader'>


In [26]:
supplied_data_reader.dataset()

<xarray.Dataset> Size: 4GB
Dimensions:                (init: 8, lead: 4, member: 11, lat: 192, lon: 384,
                            lev: 39)
Coordinates:
  * init                   (init) datetime64[ns] 64B 1994-05-01 ... 1997-11-01
  * lead                   (lead) int64 32B 0 1 2 3
  * member                 (member) int64 88B 0 1 2 3 4 5 6 7 8 9 10
  * lat                    (lat) float64 2kB 89.28 88.36 87.42 ... -88.36 -89.28
  * lon                    (lon) float64 3kB 0.0 0.9375 1.875 ... 358.1 359.1
  * lev                    (lev) float64 312B 0.01 0.02 0.04 ... 975.0 1e+03
    atmosphereSingleLayer  float64 8B ...
    heightAboveGround      float64 8B ...
    meanSea                float64 8B ...
    nominalTop             float64 8B ...
    surface                float64 8B ...
Data variables:
    tprs                   (init, lead, member, lat, lon, lev) float32 4GB dask.array<chunksize=(1, 1, 11, 192, 384, 1), meta=np.ndarray>
Attributes:
    Conventions:             CF-1.7
    GRIB_centre:             kwbc
    GRIB_centreDescription:  US National Weather Service - NCEP
    history:                 2024-12-09T11:56 GRIB to CDM+CF via cfgrib-0.9.1...
    institution:             US National Weather Service - NCEP

In [27]:
supplied_data_reader.dataset().init.values

array(['1994-05-01T00:00:00.000000000', '1994-11-01T00:00:00.000000000',
       '1995-05-01T00:00:00.000000000', '1995-11-01T00:00:00.000000000',
       '1996-05-01T00:00:00.000000000', '1996-11-01T00:00:00.000000000',
       '1997-05-01T00:00:00.000000000', '1997-11-01T00:00:00.000000000'],
      dtype='datetime64[ns]')

Why do this?  Internally in the `ufs-analysis` source code, the `SUPPLIED_DataReader` subclass offers many conveniences when manipulating data in long and convoluted analyses.  Additionally, from the user's perspective, there should be a way to make use of the `DataReader` class with *any* Xarray dataset that the user wishes, so that they are not strictly limited to what is stored in UFS and ERA5 cloud buckets.<br>

<h2>WINDS</h2>

There is one more difference between `SUPPLIED` data readers and `UFS`/`ERA5` data readers, and that is hardcoded metadata related to complimentary data fields.  For reasons that will become more clear in the Regridding section of this tutorial, `UFS` and `ERA5` data readers are hardcoded with knowledge of U-V wind fields, while `SUPPLIED` data readers are not.

In [28]:
era5_data_reader.WINDS

[{'U_WIND': 'u_component_of_wind', 'V_WIND': 'v_component_of_wind'},
 {'U_WIND': '10m_u_component_of_wind', 'V_WIND': '10m_v_component_of_wind'}]

In [29]:
ufs_data_reader.WINDS

[{'U_WIND': 'uprs', 'V_WIND': 'vprs'},
 {'U_WIND': 'u', 'V_WIND': 'v'},
 {'U_WIND': 'u10m', 'V_WIND': 'v10m'}]

`WINDS` metadata are a list of dictionaries in which each dictionary has 2 keys, one called `U_WIND` and the other `V_WIND`, with values corresponding to respective field names found in UFS and ERA5 datasets.  These `WINDS` are not necessarily exhaustive since different experiments, models, or data publishings may have evolving variable names.  The `WINDS` that we have hardcoded at this time are simply the fields that were most relevant during development.  If the user needs to adjust `WINDS` in a pinch, they can do it like so:

In [30]:
ufs_data_reader.WINDS = [{'U_WIND': 'dummy_u_wind', 'V_WIND': 'dummy_v_wind'}]

In [31]:
ufs_data_reader.WINDS

[{'U_WIND': 'dummy_u_wind', 'V_WIND': 'dummy_v_wind'}]

Again, in the next section (Regridding), we'll show you why the WINDS metadata is helpful.

<h2>Updating the DataReader</h2>

Another method of convenience is the `DataReader.update()` method.  The `.update()` method allows you to override a data reader's underlying dataset with any Xarray dataset you have in memory.  While this may seem redundant considering the `SUPPLIED` data reader subclass, again, it has many use cases in this repository's analysis codes and notebooks, especially where Regridding is concerned. 

In [32]:
ufs_data_reader.update(ds=ds)

Dataset updated.


In [33]:
ufs_data_reader.dataset()

<xarray.Dataset> Size: 4GB
Dimensions:                (init: 8, lead: 4, member: 11, lat: 192, lon: 384,
                            lev: 39)
Coordinates:
  * init                   (init) datetime64[ns] 64B 1994-05-01 ... 1997-11-01
  * lead                   (lead) int64 32B 0 1 2 3
  * member                 (member) int64 88B 0 1 2 3 4 5 6 7 8 9 10
  * lat                    (lat) float64 2kB 89.28 88.36 87.42 ... -88.36 -89.28
  * lon                    (lon) float64 3kB 0.0 0.9375 1.875 ... 358.1 359.1
  * lev                    (lev) float64 312B 0.01 0.02 0.04 ... 975.0 1e+03
    atmosphereSingleLayer  float64 8B ...
    heightAboveGround      float64 8B ...
    meanSea                float64 8B ...
    nominalTop             float64 8B ...
    surface                float64 8B ...
Data variables:
    tprs                   (init, lead, member, lat, lon, lev) float32 4GB dask.array<chunksize=(1, 1, 11, 192, 384, 1), meta=np.ndarray>
Attributes:
    Conventions:             CF-1.7
    GRIB_centre:             kwbc
    GRIB_centreDescription:  US National Weather Service - NCEP
    history:                 2024-12-09T11:56 GRIB to CDM+CF via cfgrib-0.9.1...
    institution:             US National Weather Service - NCEP

In [34]:
ufs_data_reader.dataset().init.values

array(['1994-05-01T00:00:00.000000000', '1994-11-01T00:00:00.000000000',
       '1995-05-01T00:00:00.000000000', '1995-11-01T00:00:00.000000000',
       '1996-05-01T00:00:00.000000000', '1996-11-01T00:00:00.000000000',
       '1997-05-01T00:00:00.000000000', '1997-11-01T00:00:00.000000000'],
      dtype='datetime64[ns]')

As you can see, after updating the `ufs_data_reader` object with a previously generated subset, its underlying dataset only includes the `tprsr` field and a limited number of `inits`.  Note that for the `.update()` method work, your subset must still retain the general characteristics of a model or verification dataset. For example, it must have latitude-longitude coordinates and either `time` or `init` + `lead` coordinates.

<h2>DataSet requirements:</h2>

Any dataset that you supply must meet the following requirements:
- The dataset must have a temporal coordinate, either<br>
`time` of `datetime64` type, OR<br>
`init` of `datetime64` type + `lead` of `int64` type.
- If the dataset has `leads`, then that dimension must have an attribute called `units` with a value of either `months`, `days`, or `hours`.
- The dataset must have a dimension called either `latitude` or `lat`, as well as a dimension called either `longitude` or `lon`.

***Heads up:*** **This package is designed specially for UFS SFS data.**  Some model forecast data are published in such a way that each initialization has its own individual file, and within each individual dataset there is a `time` coordinate that refers to each `lead` after initialization.  In order for our `ufs-analysis` package to process these data, they must be restructured beforehand to the meet the above requirements!!!

<h2>Regridding</h2>

Alongside the `DataReader` class, the `Regrid` class is the other foundational tool for the `ufs-analysis` package.<br>
It has three principal functions:<br>
1. **Resample** (temporal)**:**  Match the temporal resolution of a verification dataset to the UFS dataset's lead resolution.
2. **Regrid** (spatial)**:** Interpolate a higher resolution spatial grid onto a lower resolution grid. 
3. **Align** (temporal)**:** Convert time coordinates of a verification dataset into init+lead coordinates of the UFS dataset.

Generally speaking, these function must be run in this exact order in a processing workflow, but depending on the exact nature of the data supplied, some steps may be bypassed (see `Regrid` Ruleset below).

At the end of a complete `Regrid` workflow, both datasets will have the same temporal resolution and domain, same spatial resolution, and the same `init`+`lead` coordinate structure.

The scope of the `Regrid` class is limited by design, and because these functions can be computationally expensive depending on the datasets provided, the class enforces certain logical constraints to prevent bottlenecking. 

Let's instantiate a `Regrid` object and then dig into the the details. 

In [35]:
from src.regridder import Regrid

In [36]:
# Start over from scratch and instantiate two new `DataReader` objects (as done before):
# Get SFS atmospheric model data
ufs_data_reader = dr.getDataReader(datasource='UFS',                  
                                   experiment = 'beta.0.1',  # Read the beta1 experiment for this example
                                   model='atm')

# Get ERA5 Analysis-Ready data
era5_data_reader = dr.getDataReader(datasource='ERA5')


No filename provided; deferring to default
Reading data from s3://noaa-oar-sfsdev-pds/experiments/phase_1/beta.0.1/atm_monthly.zarr
No filename provided; deferring to default
Reading data from gs://gcp-public-data-arco-era5/ar/1959-2022-6h-512x256_equiangular_conservative.zarr


In [37]:
# Instantiate Regrid object:
regridder = Regrid.Regrid(data_reader1=ufs_data_reader,                                                              
                          data_reader2=era5_data_reader,                                                            
                          method='linear')


Regrid Object initialized.

___Resample Instructions___
data_reader2 must be temporally resampled before spatially regridding.
Resample these data by running <RegridObj>.resample(var=<var>, lev=<lev>, time=<time_range>)
To see all variables available for resample, run <RegridObj>.resample_vars()

___Regrid Instructions___
Initialized the Regridder with method 'linear'
Input grid shape data_reader1 (UFS_DataReader): lat 361, lon 720
Output grid shape data_reader2 (ERA5_DataReader): lat 256, lon 512
Regrid these data by running <RegridObj>.regrid(var=<var>, lev=<lev>, time=<time_range>)
To see all variables available for regrid, run <RegridObj>.regrid_vars()

___Align Instructions___
data_reader2 can have its time coordinates converted to init+lead.
Lead resolution of the UFS dataset interpreted as monthly intervals.
Align these data by running <RegridObj>.align() (You may need to resample and/or regrid first.)



Spatial interpolation is performed via Python's `scipy` package.  The following interpolation methods are possible: `linear`, `nearest`, `slinear`, `cubic`, `quintic`, and `pchip`.

The above printout summarizes the logical state of your `Regrid` object, including helpful API hints for your processing workflow.<br>
Here is the complete set of rules:

**`Regrid` Ruleset:**<br>
1. The `Regrid` class operates on two `DataReader` objects at a time.
2. At least one `DataReader` must be UFS-type.
3. UFS-type `DataReader` objects are assumed to have `init`+`lead` temporal coordinate system.
4. Non-UFS-type `DataReader` objects are assumed to have `time` temporal coordinate system.
5. The higher resolution dataset is spatially regridded onto the lower resolution one, and never the other way around.
6. If temporal resolutions do not match, then the non-UFS-type `DataReader` must first be temporally resampled before spatial regridding is allowed to take place.
7. Similarly, a non-UFS-type `DataReader` cannot be temporally aligned before first being resampled and/or spatially regridded (if necessary).
8. Resampling is disabled altogether if both `DataReader` objects are UFS-type.
9. Only one scalar field, or two complimentary fields (e.g. WIND vectors) can be processed at a time for all three functions.
10. For UFS-type `DataReader` objects, only one member can be processed at a time, or alternatively an ensemble average.
11. For fields with a vertical dimension, only one level can be processed at a time.

For `.resample()` and `.regrid()` methods, the user must specify which variable to process. Because the resolutions of each dataset affect the directionality of these processes, the following helper functions clarify which data are available for each step:

In [38]:
regridder.resample_vars()

Variables available for Resample (data_reader2):
                                                                                      
10m_u_component_of_wind               standard_deviation_of_filtered_subgrid_orography
10m_v_component_of_wind               standard_deviation_of_orography                 
10m_wind_speed                        surface_pressure                                
2m_temperature                        temperature                                     
angle_of_sub_gridscale_orography      toa_incident_solar_radiation                    
anisotropy_of_sub_gridscale_orography toa_incident_solar_radiation_12hr               
geopotential                          toa_incident_solar_radiation_24hr               
geopotential_at_surface               toa_incident_solar_radiation_6hr                
high_vegetation_cover                 total_cloud_cover                               
lake_cover                            total_column_water_vapour                  

In [39]:
regridder.regrid_vars()

Variables available for Regrid (data_reader1):
                                           
SUNSDsfc     dp2m      pratesfc   suswrfsfc
acpcpsfc     duvbsfc   prmsl      suswrftoa
aptmp        fldcpsfc  pwat       t        
avg_alsfc    fricvsfc  q          tcc      
avg_hcc      fsrsfc    rh2m       tmax     
avg_ishfsfc  gfluxsfc  sdesfc     tmin     
avg_lcc      gh        sdlwrfsfc  tmp2m    
avg_mcc      gustsfc   sdswrfsfc  tozne    
avg_slhtfsfc hindexsfc sdswrftoa  tpsfc    
avg_utauasfc hlcy      sdwesfc    tsfc     
avg_vtauasfc iegwsssfc sithicksfc u        
btmptoa      ingwsssfc snohfsfc   u10m     
capesfc      lftxsfc   snowcsfc   v        
cduvbsfc     lftxsfc4  soill      v10m     
cinsfc       lsmsfc    soilw      vbdsfsfc 
cisfc        max_10si  spfh2m     vissfc   
cnwatsfc     mslet     spsfc      watrsfc  
cpofpsfc     nbdsfsfc  ssrunsfc   wiltsfc  
cprsfc       ncpcpsfc  st                  
cwat         o3mr      sulwrfsfc           
cwork        orogsfc   sulwrf

This ERA5 dataset is published on an hourly basis, while the UFS dataset has leads representing monthly averages.  The UFS dataset, however, has a higher spatial resolution.  Therefore, we must first resample ERA5, then spatially regrid UFS, before performing the final alignment.

In this example, let's work with 2-meter temperature fields.  This is a flat scalar variable that will be quickest to process. 

In [40]:
# Print underlying Xarray object for reference
era5_data_reader.dataset()

<xarray.Dataset> Size: 5TB
Dimensions:                                           (time: 92044, lon: 512,
                                                       lat: 256, lev: 13)
Coordinates:
  * time                                              (time) datetime64[ns] 736kB ...
  * lon                                               (lon) float64 4kB 0.0 ....
  * lat                                               (lat) float64 2kB 89.65...
  * lev                                               (lev) int64 104B 50 ......
Data variables: (12/38)
    10m_u_component_of_wind                           (time, lat, lon) float32 48GB dask.array<chunksize=(124, 256, 512), meta=np.ndarray>
    10m_v_component_of_wind                           (time, lat, lon) float32 48GB dask.array<chunksize=(124, 256, 512), meta=np.ndarray>
    10m_wind_speed                                    (time, lat, lon) float32 48GB dask.array<chunksize=(124, 256, 512), meta=np.ndarray>
    2m_temperature                                    (time, lat, lon) float32 48GB dask.array<chunksize=(124, 256, 512), meta=np.ndarray>
    angle_of_sub_gridscale_orography                  (lat, lon) float32 524kB dask.array<chunksize=(256, 512), meta=np.ndarray>
    anisotropy_of_sub_gridscale_orography             (lat, lon) float32 524kB dask.array<chunksize=(256, 512), meta=np.ndarray>
    ...                                                ...
    type_of_high_vegetation                           (lat, lon) float32 524kB dask.array<chunksize=(256, 512), meta=np.ndarray>
    type_of_low_vegetation                            (lat, lon) float32 524kB dask.array<chunksize=(256, 512), meta=np.ndarray>
    u_component_of_wind                               (time, lat, lon, lev) float32 627GB dask.array<chunksize=(124, 256, 512, 13), meta=np.ndarray>
    v_component_of_wind                               (time, lat, lon, lev) float32 627GB dask.array<chunksize=(124, 256, 512, 13), meta=np.ndarray>
    vertical_velocity                                 (time, lat, lon, lev) float32 627GB dask.array<chunksize=(124, 256, 512, 13), meta=np.ndarray>
    wind_speed                                        (time, lat, lon, lev) float32 627GB dask.array<chunksize=(124, 256, 512, 13), meta=np.ndarray>

In [41]:
# Print underlying Xarray object for reference
ufs_data_reader.dataset()

<xarray.Dataset> Size: 3TB
Dimensions:       (init: 60, member: 11, lead: 12, lat: 361, lon: 720, lev: 33)
Coordinates:
  * init          (init) datetime64[ns] 480B 1994-05-01 ... 2023-11-01
  * member        (member) int64 88B 0 1 2 3 4 5 6 7 8 9 10
  * lead          (lead) int64 96B 0 1 2 3 4 5 6 7 8 9 10 11
  * lat           (lat) float64 3kB 90.0 89.5 89.0 88.5 ... -89.0 -89.5 -90.0
  * lon           (lon) float64 6kB 0.0 0.5 1.0 1.5 ... 358.0 358.5 359.0 359.5
  * lev           (lev) int64 264B 0 1 2 3 5 7 10 ... 850 900 925 950 975 1000
Data variables: (12/81)
    acpcpsfc      (init, lead, member, lat, lon) float32 8GB dask.array<chunksize=(1, 1, 11, 361, 720), meta=np.ndarray>
    SUNSDsfc      (init, lead, member, lat, lon) float32 8GB dask.array<chunksize=(1, 1, 11, 361, 720), meta=np.ndarray>
    avg_hcc       (init, lead, member, lat, lon) float32 8GB dask.array<chunksize=(1, 1, 11, 361, 720), meta=np.ndarray>
    avg_ishfsfc   (init, lead, member, lat, lon) float32 8GB dask.array<chunksize=(1, 1, 11, 361, 720), meta=np.ndarray>
    avg_mcc       (init, lead, member, lat, lon) float32 8GB dask.array<chunksize=(1, 1, 11, 361, 720), meta=np.ndarray>
    avg_lcc       (init, lead, member, lat, lon) float32 8GB dask.array<chunksize=(1, 1, 11, 361, 720), meta=np.ndarray>
    ...            ...
    v             (init, lead, member, lat, lon, lev) float32 272GB dask.array<chunksize=(1, 1, 11, 361, 720, 1), meta=np.ndarray>
    v10m          (init, lead, member, lat, lon) float32 8GB dask.array<chunksize=(1, 1, 11, 361, 720), meta=np.ndarray>
    watrsfc       (init, lead, member, lat, lon) float32 8GB dask.array<chunksize=(1, 1, 11, 361, 720), meta=np.ndarray>
    wiltsfc       (init, lead, member, lat, lon) float32 8GB dask.array<chunksize=(1, 1, 11, 361, 720), meta=np.ndarray>
    vissfc        (init, lead, member, lat, lon) float32 8GB dask.array<chunksize=(1, 1, 11, 361, 720), meta=np.ndarray>
    vbdsfsfc      (init, lead, member, lat, lon) float32 8GB dask.array<chunksize=(1, 1, 11, 361, 720), meta=np.ndarray>

In [42]:
# Resample ERA5 data from 1994 to the end of the data record.
regridder.resample(var='2m_temperature', lev=None, time=('1994-05-01', '2021-12-31T18'), use_mp=True)

Resampling data_reader2 data to 'MS' using mean aggregation.
Resampling from 1994-05-01T00 to 2021-12-31T18
Number of cores available: 2
Finished multiprocessing.  Concatenating results.
Resample completed in 1.77 minutes.
Resample results stored in <RegridObj>.resampled


Note that, in this case, we could have also specified time like `time=('1994-05-01', None)`, where the `None` would automatically get all data to the end of the temporal record.

The `.resample()` method can be the most expensive function to run, especially since this is normally when data are loaded into memory for the first time.  Additionally, due to the nature of `.zarr`-formatted data, IO can be prohibitively expensive when dealing with fields with vertical dimensions. `.resample()` will therefore attempt to utilize multiple computing cores if they are available.  This can be controled with the `use_mp` option which is set to `True` by default.  Make sure to set the `time` parameter so that only relevant data are considered, if necessary.<br>

After running the `.resample()` method, the results are stored as new `SUPPLIED_DataReader` objects!

In [43]:
regridder.resampled

In [44]:
# Metadata is preserved
regridder.resampled.WINDS

[{'U_WIND': 'u_component_of_wind', 'V_WIND': 'v_component_of_wind'},
 {'U_WIND': '10m_u_component_of_wind', 'V_WIND': '10m_v_component_of_wind'}]

We can interact with this DataReader object in all the same ways already discussed in the DataReader section of this notebook.<br>
Let's just look at the underlying Xarray object for now:

In [45]:
regridder.resampled.dataset()

<xarray.Dataset> Size: 174MB
Dimensions:         (time: 332, lat: 256, lon: 512)
Coordinates:
  * time            (time) datetime64[ns] 3kB 1994-05-01 ... 2021-12-01
  * lat             (lat) float64 2kB 89.65 88.95 88.24 ... -88.24 -88.95 -89.65
  * lon             (lon) float64 4kB 0.0 0.7031 1.406 ... 357.9 358.6 359.3
Data variables:
    2m_temperature  (time, lat, lon) float32 174MB 263.1 263.1 ... 247.5 247.5

We just resampled our ERA5 data into monthly averages per the UFS dataset and have a matching temporal record beginning in 1994.<br>
Now we can spatially regrid our UFS dataset.  Let's run this analysis on an ensemble average of all members (else, we'd need to specify something like `member=3` to process 1 member at a time).

In [46]:
regridder.regrid(var='tmp2m', ens_avg=True)

Regridding data_reader1 grid (UFS_DataReader) onto data_reader2 grid (ERA5_DataReader)
Taking Ensemble Average
Running scalar regrid on tmp2m
Completed scalar regrid in 3.31 minutes.
Regrid results stored in <RegridObj>.regridded


The results of `.regrid()` are stored in the same manner as the resampled results:

In [47]:
regridder.regridded

In [48]:
regridder.regridded.dataset()

<xarray.Dataset> Size: 755MB
Dimensions:  (init: 60, lead: 12, lat: 256, lon: 512)
Coordinates:
  * init     (init) datetime64[ns] 480B 1994-05-01 1994-11-01 ... 2023-11-01
  * lead     (lead) int64 96B 0 1 2 3 4 5 6 7 8 9 10 11
  * lat      (lat) float64 2kB 89.65 88.95 88.24 87.54 ... -88.24 -88.95 -89.65
  * lon      (lon) float64 4kB 0.0 0.7031 1.406 2.109 ... 357.9 358.6 359.3
Data variables:
    tmp2m    (init, lead, lat, lon) float64 755MB 264.7 264.7 ... 236.5 236.5
Attributes:
    description:  regrid

Lastly, we can run the final align step!  This will convert ERA5 temporal coordinates to exactly match the `init`+`lead` domain of the UFS dataset.

In [49]:
regridder.align()

Aligning time coordinate to init+lead coordinates.
MODEL_LEADS: [ 0  1  2  3  4  5  6  7  8  9 10 11]
Time dimensions aligned:  matched 658 timesteps
Align results stored in <RegridObj>.aligned


/home/thamzey/tutorial/./ufs-analysis/src/regridder/Regrid.py:805: UserWarning: Verif data not available for 14 timesteps, filled with NaN.
  warnings.warn(msg)


When running .align(), you may see a warning message appear regarding NaN filling.  This is perfectly okay; what it means is that the temporal domain of the UFS dataset exceeds that of the ERA5 verification dataset, so in order to exactly match the `init`s and `leads` in the UFS dataset, `nan` filler data are inserted.

Et voila, you have completed a `Regrid` workflow!  In this case, the final Xarray datasets are stored here:

In [50]:
# Fully processed ERA5 DataReader
regridder.aligned

In [51]:
# Fully processed ERA5 dataset:
regridder.aligned.dataset()

<xarray.Dataset> Size: 352MB
Dimensions:         (lead: 12, init: 56, lat: 256, lon: 512)
Coordinates:
  * lead            (lead) int64 96B 0 1 2 3 4 5 6 7 8 9 10 11
    time            (init, lead) datetime64[us] 5kB 1994-05-01 ... 2022-10-01
  * init            (init) datetime64[ns] 448B 1994-05-01 ... 2021-11-01
  * lat             (lat) float64 2kB 89.65 88.95 88.24 ... -88.24 -88.95 -89.65
  * lon             (lon) float64 4kB 0.0 0.7031 1.406 ... 357.9 358.6 359.3
Data variables:
    2m_temperature  (init, lead, lat, lon) float32 352MB 263.1 263.1 ... nan nan
Attributes:
    model_freq_unit:      MS
    lead_step:            30 days
    alignment_tolerance:  16 days

In [52]:
# Fully processed UFS DataReader:
regridder.regridded

In [53]:
# Fully processed UFS dataset:
regridder.regridded.dataset()

<xarray.Dataset> Size: 755MB
Dimensions:  (init: 60, lead: 12, lat: 256, lon: 512)
Coordinates:
  * init     (init) datetime64[ns] 480B 1994-05-01 1994-11-01 ... 2023-11-01
  * lead     (lead) int64 96B 0 1 2 3 4 5 6 7 8 9 10 11
  * lat      (lat) float64 2kB 89.65 88.95 88.24 87.54 ... -88.24 -88.95 -89.65
  * lon      (lon) float64 4kB 0.0 0.7031 1.406 2.109 ... 357.9 358.6 359.3
Data variables:
    tmp2m    (init, lead, lat, lon) float64 755MB 264.7 264.7 ... 236.5 236.5
Attributes:
    description:  regrid

In other cases, you may want to process WIND vectors that have vertical dimensions.<br>
Here is a quick example for doing so, starting from scratch:

In [54]:
regridder = Regrid.Regrid(data_reader1=ufs_data_reader,                                                              
                          data_reader2=era5_data_reader,                                                            
                          method='linear')


Regrid Object initialized.

___Resample Instructions___
data_reader2 must be temporally resampled before spatially regridding.
Resample these data by running <RegridObj>.resample(var=<var>, lev=<lev>, time=<time_range>)
To see all variables available for resample, run <RegridObj>.resample_vars()

___Regrid Instructions___
Initialized the Regridder with method 'linear'
Input grid shape data_reader1 (UFS_DataReader): lat 361, lon 720
Output grid shape data_reader2 (ERA5_DataReader): lat 256, lon 512
Regrid these data by running <RegridObj>.regrid(var=<var>, lev=<lev>, time=<time_range>)
To see all variables available for regrid, run <RegridObj>.regrid_vars()

___Align Instructions___
data_reader2 can have its time coordinates converted to init+lead.
Lead resolution of the UFS dataset interpreted as monthly intervals.
Align these data by running <RegridObj>.align() (You may need to resample and/or regrid first.)



In [55]:
# Resample ERA5 data from 1994 to the end of the data record.
regridder.resample(var=['u_component_of_wind', 'v_component_of_wind'], lev=500, time=('2020-05-01', '2021-12-31T18'), use_mp=True)

Resampling data_reader2 data to 'MS' using mean aggregation.
Resampling from 2020-05-01T00 to 2021-12-31T18
Number of cores available: 2
Finished multiprocessing.  Concatenating results.
Resample completed in 1.67 minutes.
Resample results stored in <RegridObj>.resampled


WIND fields are interpolated differently than scalar fields.  Earlier in this tutorial, we mentioned how WINDS metadata are useful for regridding.  If the `Regrid` object notices that you're interpolating WIND vector fields, then the `method=linear` parameter is totally ignored.  Instead, ***WIND vectors always interpolated via spherical harmonic analysis.***  

In [56]:
regridder.regrid(var=['u', 'v'], lev=500, time=('2020-05-01', '2021-11-01'), ens_avg=True)

Regridding data_reader1 grid (UFS_DataReader) onto data_reader2 grid (ERA5_DataReader)
Taking Ensemble Average
Running spherical harmonics on u and v
Completed sperical harmonics in 0.01 minutes.
Regrid results stored in <RegridObj>.regridded


In [57]:
regridder.align()

Aligning time coordinate to init+lead coordinates.
MODEL_LEADS: [ 0  1  2  3  4  5  6  7  8  9 10 11]
Time dimensions aligned:  matched 34 timesteps
Align results stored in <RegridObj>.aligned


/home/thamzey/tutorial/./ufs-analysis/src/regridder/Regrid.py:805: UserWarning: Verif data not available for 14 timesteps, filled with NaN.
  warnings.warn(msg)


In [58]:
regridder.regridded.dataset()[['u', 'v']]

<xarray.Dataset> Size: 50MB
Dimensions:  (init: 4, lead: 12, lev: 1, lat: 256, lon: 512)
Coordinates:
  * init     (init) datetime64[ns] 32B 2020-05-01 2020-11-01 ... 2021-11-01
  * lead     (lead) int64 96B 0 1 2 3 4 5 6 7 8 9 10 11
  * lev      (lev) int64 8B 500
  * lat      (lat) float64 2kB 89.65 88.95 88.24 87.54 ... -88.24 -88.95 -89.65
  * lon      (lon) float64 4kB 0.0 0.7031 1.406 2.109 ... 357.9 358.6 359.3
Data variables:
    u        (init, lead, lev, lat, lon) float32 25MB -0.6965 -0.731 ... 1.199
    v        (init, lead, lev, lat, lon) float32 25MB -2.821 -2.812 ... -0.863
Attributes:
    description:  regrid

In [59]:
regridder.aligned.dataset()[['u_component_of_wind', 'v_component_of_wind']]

<xarray.Dataset> Size: 50MB
Dimensions:              (init: 4, lead: 12, lat: 256, lon: 512, lev: 1)
Coordinates:
  * init                 (init) datetime64[ns] 32B 2020-05-01 ... 2021-11-01
  * lead                 (lead) int64 96B 0 1 2 3 4 5 6 7 8 9 10 11
    time                 (init, lead) datetime64[us] 384B 2020-05-01 ... 2022...
  * lat                  (lat) float64 2kB 89.65 88.95 88.24 ... -88.95 -89.65
  * lon                  (lon) float64 4kB 0.0 0.7031 1.406 ... 358.6 359.3
  * lev                  (lev) int64 8B 500
Data variables:
    u_component_of_wind  (init, lead, lat, lon, lev) float32 25MB -3.407 ... nan
    v_component_of_wind  (init, lead, lat, lon, lev) float32 25MB -0.6814 ......
Attributes:
    model_freq_unit:      MS
    lead_step:            30 days
    alignment_tolerance:  16 days

<h2>Common mistakes one could make when regridding</h2>

As previously mentioned, the `Regrid` class is tightly controlled to prevent erroneous results and/or excessive computations.  A series of logic checks must be passed before any interpolation is done.  The user will be presented with meaningful error messages if something is wrong.  Here are some examples:

You supplied one wind vector v but not the other, so spherical harmonics cannot be run:

In [60]:
try:
    regridder.regrid(var=['v'], lev=500, time=('2020-05-01', '2021-11-01'), ens_avg=True)
except Exception as e:
    print(str(e))

Regridding data_reader1 grid (UFS_DataReader) onto data_reader2 grid (ERA5_DataReader)
You supplied one wind vector v but not the other, so spherical harmonics cannot be run.


You must specify a single vertical level to regrid:

In [61]:
try:
    regridder.regrid(var=['u', 'v'], time=('2020-05-01', '2021-11-01'), ens_avg=True)
except Exception as e:
    print(str(e))

Regridding data_reader1 grid (UFS_DataReader) onto data_reader2 grid (ERA5_DataReader)
Taking Ensemble Average
'You must specify a single vertical level to regrid.'


To regrid an ensemble model, you must specify a member or set ens_avg=True

In [62]:
try:
    regridder.regrid(var=['u', 'v'], lev=500, time=('2020-05-01', '2021-11-01'), ens_avg=False)
except Exception as e:
    print(str(e))

Regridding data_reader1 grid (UFS_DataReader) onto data_reader2 grid (ERA5_DataReader)
To regrid an ensemble model, you must specify a member or set ens_avg=True


Spherical harmonics can only be run on the full global domain.
If you want to regrid wind vector data, do not slice by lat-lon yet.

In [63]:
try:
    regridder.regrid(var=['u', 'v'], lev=500, time=('2020-05-01', '2021-11-01'), ens_avg=True, lat=(30, -30))
except Exception as e:
    print(str(e))

Regridding data_reader1 grid (UFS_DataReader) onto data_reader2 grid (ERA5_DataReader)
Spherical harmonics can only be run on the full global domain.
If you want to regrid wind vector data, do not slice by lat-lon yet.


Similar logic checks exist for the `.resample()` method as well!

<h2>Conclusion</h2>

In this tutorial, you learned about the `ufs-community/ufs-analysis` Python package and how to use two of its core modules: `datareader` and `regridder`. You learned how to use the `DataReader` class to query both UFS model and ERA5 verification datasets from online data repositories and how to retrieve subsets of your data.  You also learned how to resample, regrid, and align datasets, opening the door to further analysis opportunities and direct point-by-point comparisons between model forecasts and verification/observations.

This package is still in relative infancy and subject to change. While efforts will be made to keep the existing API as-is, new features and functionality will be added, especially to ensure compatibility with different grid types (e.g. tripolar grids) as the next generation of coupled modeling systems come into the fore.

Further API documentation is still in the works.  Keep up-to-date with the `ufs-analysis` repository on Github (https://github.com/ufs-community/ufs-analysis) for news and information!  